# LEGACY

## DATASET

In [ ]:
caption_dir = '../datasets/Captioning/captions.txt'
lines = []
with open(caption_dir,'r') as file :
    lines = [line.split(',') for line in file.readlines()]
    lines = [ [ line[:-1][0], line[-1].replace('\n','') ] for line in lines]
# captions = lines[:,1]

lines.pop(0)
corpus = ''
for line in lines :
    # print(line[1])
    corpus += " "
    corpus+=str(line[1])



caption_tokeniser = CaptionTokeniser()
caption_tokeniser.create_vocab(corpus)

caption_dict = {}

for line in lines : 
    jpg_name = line[0]
    caption = str(line[1])
    sent = []
    for word in word_tokenize(caption.lower()):
        sent.append(word) if word.isalpha() else None
    encoded_caption = caption_tokeniser.encode_arr(sent)
    if jpg_name in caption_dict:
        caption_dict[jpg_name].append(encoded_caption)
    else:
        caption_dict[jpg_name] = [encoded_caption]



In [ ]:
image_dir = Path('../datasets/Captioning/Images')
jpegs = [f.name for f in image_dir.glob('*.jpg')]


dataset = FeatureDataset(image_dir, jpegs,caption_dict, caption_tokeniser.vocab['<BOS>'], caption_tokeniser.vocab['<EOS>'] )


## DECODER BLOCK


In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, embed_dim, attention_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.MaskedAttention = MultiHeadAttention(embed_dim, attention_heads, masked=True)
        self.fnn = nn.Sequential(
            nn.Linear(embed_dim, 2048),
            nn.ReLU(),
            nn.Linear(2048,embed_dim)
        )
        self.layerNorm1 = nn.LayerNorm(self.embed_dim)
        self.layerNorm2 = nn.LayerNorm(self.embed_dim)

    def forward(self, input_embeds): 
        o = input_embeds + self.MaskedAttention(input_embeds) # (B,T,E)
        o = self.layerNorm1(o) # (B,T,E)
        o = o + self.fnn(o) # (B,T,E)
        o = self.layerNorm2(o) # (B,T,E)
        return o # (B,T,E)

In [ ]:
class SimpleDecoder(nn.Module):
    def __init__(self, vocab_count, embedding_dim,attention_heads, n_blocks):
        super().__init__()
        self.n_blocks = n_blocks
        self.embedding = nn.Embedding(vocab_count, embedding_dim)
        self.add_positional = AddPositionalEmbeds()
        self.onn = nn.Sequential(
                    nn.Linear(embedding_dim, vocab_count), #(B,T,E)
                    # nn.Softmax(dim = -1)
        )

        self.decoders = nn.ModuleList(DecoderBlock(embedding_dim, attention_heads) for _ in range(n_blocks))

        

    def forward(self,batch):
        B,T = batch.shape
        embeds = self.embedding(batch) # shape = (B,T,E)
        embeds = self.add_positional(embeds) 
        o = embeds # (B,T,E)
        for block in self.decoders :
            o = block(o) # (B,T,E)
        o = self.onn(o)
        return o
        
        

# START

In [59]:
import torch
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader
from nltk import word_tokenize, sent_tokenize
from collections import Counter
import math
from transformers import AutoFeatureExtractor, ResNetForImageClassification
from datasets import load_dataset
from torchinfo import summary
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
import numpy as np 

# TOKENIZER


In [42]:
class CaptionTokeniser :
    def __init__(self,  max_vocab_size = None):
        self.max_vocab_size = max_vocab_size
        
        self.word_to_idx = {
            "<PAD>": 0,
            "<BOS>": 1,
            "<EOS>": 2,
            "<UNK>": 3
        }


        self.vocab = self.word_to_idx
        self.idx_to_word = {
            0 : "<PAD>",
            1 : "<BOS>",
            2 : "<EOS>",
            3 : "<UNK>"
        }

    def create_vocab(self,text):
        tokens = word_tokenize(text.lower())
        tokens = [
            word for word in tokens
            if word.isalpha()
        ]
        word_counts = Counter(tokens)
        for word, count in word_counts.most_common():
            # if count >= 2:
            self.word_to_idx[word] = len(self.word_to_idx)
            self.idx_to_word[len(self.idx_to_word)] = word


    def add_to_vocab(self,text):
        words = [word for word in word_tokenize(text.lower())
                 if word.isalpha()]
        candidate_vocab = set(words)
        for word in candidate_vocab :
            if word not in self.vocab.keys():
                self.word_to_idx[word] = len(self.vocab.keys())
                self.idx_to_word[len(self.idx_to_word.keys())] = word


    def encode_arr(self, arr):
        out = [self.word_to_idx[word] for word in arr]
        return out

    def decode_arr(self, idx_arr):
        out = [self.idx_to_word[idx] for idx in idx_arr]
        return out

    

# FULL PIPELINE

In [43]:
feature_extractor = AutoFeatureExtractor.from_pretrained("microsoft/resnet-34")
model = ResNetForImageClassification.from_pretrained("microsoft/resnet-34")

spatial_feature_encoder = nn.Sequential(
    *list(model.children())[:-1]
)

test_tensor = torch.rand(10,3,244,244)
with torch.no_grad():
    y = spatial_feature_encoder(test_tensor).last_hidden_state
print(y.shape)

torch.Size([10, 512, 8, 8])


In [44]:
class FeatureDataset(Dataset):
    def __init__(self, image_dir, jpg_list, caption_dict,BOS_index, EOS_index ):
        super().__init__()

        self.samples = []
        self.image_dir = image_dir
        for jpg_name in jpg_list:
            
            captions = caption_dict[jpg_name]
            
            for caption in captions :
                input_caption = [BOS_index, *caption]
                target_caption = [*caption, EOS_index]
                self.samples.append(
                    (jpg_name, input_caption, target_caption)
                )
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        (jpg_name, input_caption, target_caption) = self.samples[idx]
        jpg_path = self.image_dir/jpg_name
        jpg_image = Image.open(jpg_path).convert('RGB')
        with torch.no_grad():
            features = feature_extractor(jpg_image, return_tensors="pt").pixel_values.squeeze(0)

        input_caption = torch.tensor(input_caption , dtype=torch.long)
        target_caption = torch.tensor(target_caption , dtype=torch.long)
        return features , input_caption, target_caption


In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    images, input_captions, target_captions = zip(*batch)

    images = torch.stack(images)

    input_captions = pad_sequence(
        input_captions,
        batch_first=True,
        padding_value=caption_tokeniser.word_to_idx['<PAD>']
    )

    target_captions = pad_sequence(
        target_captions,
        batch_first=True,
        padding_value=caption_tokeniser.word_to_idx['<PAD>']
    )

    return images, input_captions, target_captions


class DataPrepPipeline():

    def __init__(self,image_dir, caption_dir):
        self.image_dir = image_dir
        self.caption_dir = caption_dir
        self.caption_dict = None
        self.caption_tokeniser = None
        self.jpegs = [f.name for f in self.image_dir.glob('*.jpg')]



    def tokenise(self):
        txt = self.caption_dir
        lines = []
        with open(txt,'r') as file :
            lines = [line.split(',') for line in file.readlines()]
            lines = [ [ line[:-1][0], line[-1].replace('\n','') ] for line in lines]
        # captions = lines[:,1]

        lines.pop(0)
        corpus = ''
        for line in lines :
            # print(line[1])
            corpus += " "
            corpus+=str(line[1])



        caption_tokeniser = CaptionTokeniser()
        caption_tokeniser.create_vocab(corpus)

        caption_dict = {}

        for line in lines : 
            jpg_name = line[0]
            caption = str(line[1])
            sent = []
            for word in word_tokenize(caption.lower()):
                sent.append(word) if word.isalpha() else None
            encoded_caption = caption_tokeniser.encode_arr(sent)
            if jpg_name in caption_dict:
                caption_dict[jpg_name].append(encoded_caption)
            else:
                caption_dict[jpg_name] = [encoded_caption]

        self.caption_dict = caption_dict
        self.caption_tokeniser = caption_tokeniser

        return caption_tokeniser, caption_dict


    def image_encoding(self , train_size, test_size, val_size):
        X = np.array(self.jpegs)
        train_jpegs, val_jpegs = train_test_split(X, test_size=val_size, shuffle=True)
        test_size_new = test_size/(1-val_size)
        train_jpegs, test_jpegs = train_test_split(train_jpegs, test_size=test_size_new, shuffle=True)

        train_dataset = FeatureDataset(image_dir, train_jpegs, self.caption_dict, self.caption_tokeniser.vocab['<BOS>'], self.caption_tokeniser.vocab['<EOS>'] )
        test_dataset = FeatureDataset(image_dir, test_jpegs, self.caption_dict, self.caption_tokeniser.vocab['<BOS>'], self.caption_tokeniser.vocab['<EOS>'] )
        val_dataset = FeatureDataset(image_dir, val_jpegs, self.caption_dict, self.caption_tokeniser.vocab['<BOS>'], self.caption_tokeniser.vocab['<EOS>'] )

        train_loader = DataLoader(
            train_dataset,
            batch_size=32,
            shuffle=True,
            collate_fn=collate_fn
        )
        test_loader = DataLoader(
            test_dataset,
            batch_size=32,
            shuffle=True,
            collate_fn=collate_fn
        )
        val_loader = DataLoader(
            dataset,
            batch_size=32,
            shuffle=True,
            collate_fn=collate_fn
        )

        return [(train_dataset, train_loader),(test_dataset, test_loader),(val_dataset, val_loader)]

(array([1, 7, 3, 2, 4]), array([6, 5]))

In [46]:
image_dir = Path('../datasets/Captioning/Images')
caption_dir = '../datasets/Captioning/captions.txt'

dataPipeline = DataPrepPipeline(image_dir, caption_dir)
caption_tokeniser, caption_dict = dataPipeline.tokenise()
dataset, loader = dataPipeline.image_encoding()

image_feature, caption_input, caption_output = next(iter(loader))

print(f'image_feature shape = {image_feature.shape}')
print(f'caption input shape = {caption_input.shape}')
print(f'caption output shape = {caption_output.shape}')

image_feature shape = torch.Size([32, 3, 224, 224])
caption input shape = torch.Size([32, 26])
caption output shape = torch.Size([32, 26])


# RESNET - ENCODER

In [47]:
class SpatialEncoder(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()

        self.RESNET_model = ResNetForImageClassification.from_pretrained("microsoft/resnet-34")
        self.last_conv_layer = [*list(self.RESNET_model.named_modules())][-8][1]
        self.feature_dim = self.last_conv_layer.out_channels

        self.spatial_feature_encoder = nn.Sequential(
            *list(self.RESNET_model.children())[:-1]
        )
        self.projection = nn.Conv2d(in_channels=self.last_conv_layer.out_channels,
                                    out_channels=embed_dim,
                                    kernel_size=1)

    def forward(self, input):
        B,C,W,H = input.shape
        print(f'ENCODER RECEIEVED SHAPE : {input.shape}')
        y = self.spatial_feature_encoder(input).last_hidden_state # (B,512,7,7) 
        print(f'SPATIAL-ENCODER RETURNED : {y.shape}')
        y = self.projection(y)
        print(f"ENCODER's Projection RETURNED : {y.shape}")
        B,C_o,W_o,H_o = y.shape
        y = y.transpose(1,3)
        y = y.reshape(B,W_o*H_o,C_o)

        return y
        


In [48]:
SPE = SpatialEncoder(20)
spatial_f, input_c, target_c = next(iter(loader))
print(spatial_f.shape)
y = SPE(spatial_f)
print(y.shape)

torch.Size([32, 3, 224, 224])
ENCODER RECEIEVED SHAPE : torch.Size([32, 3, 224, 224])
SPATIAL-ENCODER RETURNED : torch.Size([32, 512, 7, 7])
ENCODER's Projection RETURNED : torch.Size([32, 20, 7, 7])
torch.Size([32, 49, 20])


In [49]:
SPE.spatial_feature_encoder

Sequential(
  (0): ResNetModel(
    (embedder): ResNetEmbeddings(
      (embedder): ResNetConvLayer(
        (convolution): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activation): ReLU()
      )
      (pooler): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    )
    (encoder): ResNetEncoder(
      (stages): ModuleList(
        (0): ResNetStage(
          (layers): Sequential(
            (0): ResNetBasicLayer(
              (shortcut): Identity()
              (layer): Sequential(
                (0): ResNetConvLayer(
                  (convolution): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
                  (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
                  (activation): ReLU()
                )
                

In [50]:
a = [*list(SPE.RESNET_model.named_modules())]
len(a)
a[-7][-1].num_features
# a[-8][-1].out_channels
# SPE.spatial_feature_encoder

512

# MODEL - COMPONENTS

 ## ATTENTION --> MASKED-MULTI n CROSS

In [51]:
class MultiHeadAttention(nn.Module):
    def __init__(self,embed_dim, num_heads, masked = False):
        super().__init__()
        if embed_dim%num_heads != 0 :
            raise ValueError(f"cant divide embed_dim{embed_dim} into {num_heads} heads")

        self.masked = masked
        self.num_heads = num_heads
        self.q = nn.Linear(embed_dim,embed_dim)
        self.k = nn.Linear(embed_dim,embed_dim)
        self.v = nn.Linear(embed_dim,embed_dim)
        self.Wo = nn.Linear(embed_dim,embed_dim)


    def forward(self, input_batch): 
        B, T, E = input_batch.shape
        device = input_batch.device
        H = self.num_heads
        Eh = E//self.num_heads

        query_vec = self.q(input_batch)  # shape (B,T,E)x(ExE) = BxTxE
        key_vec = self.k(input_batch)    # shape (B,T,E)x(ExE) = BxTxE
        value_vec = self.v(input_batch)  # shape (B,T,E)x(ExE) = BxTxE

        head_q_vecs = query_vec.reshape(B,T,H,Eh).transpose(1,2)    #from reshape : (B,T,H,Eh), from transpose :(B,H,T,Eh) 
        head_k_vecs = key_vec.reshape(B,T,H,Eh).transpose(1,2)      #from reshape : (B,T,H,Eh), from transpose :(B,H,T,Eh)
        head_v_vecs = value_vec.reshape(B,T,H,Eh).transpose(1,2)    #from reshape : (B,T,H,Eh), from transpose :(B,H,T,Eh)

        sim_scores = head_q_vecs @ head_k_vecs.transpose(-2,- 1) # (B,H,T,Eh) . (B,H,Eh,T) = (B,H,T,T)
        sim_scores = sim_scores/math.sqrt(Eh) #(B,H,T,T)

        if self.masked :
            t_q = torch.arange(T, device=device).view(T,1)
            t_k = torch.arange(T, device=device).view(1,T)
            mask_0 = t_q >= t_k
            mask_inf = t_q < t_k 
            sim_scores = sim_scores.masked_fill(
                mask_inf,
                float('-inf')
            )         
        sim_scores = torch.softmax(sim_scores, dim = -1)#(B,H,Tq,Tk) , dim =1 , as we are softmaxing ALL KEY probs for a QUERY


        attention = sim_scores @ head_v_vecs # (B,H,T,T).(B,H,T,Eh) = (B,H,T,Eh)
        out = attention.transpose(1,2).reshape(B,T,E) # from trnaspose : (B,T,H,Eh), from reshape : (B,T,E)
        output = self.Wo(out) #(B,T,E)x(E,E) = (B,T,E)
        return output #(B,T,E)
        

In [52]:
class CrossAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, masked = False):
        super().__init__()
        self.q_w = nn.Linear(embed_dim,embed_dim)
        self.k_w = nn.Linear(embed_dim,embed_dim)
        self.v_w = nn.Linear(embed_dim,embed_dim)
        self.o_w = nn.Linear(embed_dim,embed_dim)
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.masked = masked

        
    def forward(self, batch_dec, batch_enc):    
        
        B,T_dec,E = batch_dec.shape  #shape of both batches
        _,T_enc,_ = batch_enc.shape  #shape of both batches
        H = self.num_heads
        Eh = E//H

        if E!= self.embed_dim:
            raise ValueError(f'EMBED DIM MISMATCH during instantiation : {self.embed_dim}, given batch has embed dim of {E}')
        
        if E % H != 0:
            raise ValueError(f"can't divide embed_dim {E} into {H} heads")


        query_vecs = self.q_w(batch_dec) # B,T_dec,E
        key_vecs = self.k_w(batch_enc) # B,T_enc,E
        value_vecs = self.v_w(batch_enc) # B,T_enc,E

        query_vecs = query_vecs.reshape(B,T_dec,H,Eh).transpose(1,2) #from reshape: (B,T_dec,H,Eh) , from transpose : (B,H,T_dec,Eh)
        key_vecs = key_vecs.reshape(B,T_enc,H,Eh).transpose(1,2)    #from reshape: (B,T_enc,H,Eh) , from transpose : (B,H,T_enc,Eh)
        value_vecs = value_vecs.reshape(B,T_enc,H,Eh).transpose(1,2)    #from reshape: (B,T_enc,H,Eh) , from transpose : (B,H,T_enc,Eh)

        sim_scores = query_vecs @ key_vecs.transpose(-1,-2) # (B,H,T_dec,Eh) @ (B,H,Eh,T_enc) = (B,H,T_dec,T_enc)
        if self.masked:
            T_q = torch.arange(T_dec).view((T_dec,1))
            T_k = torch.arange(T_enc).view((1,T_enc))
            mask_inf = T_k > T_q

            sim_scores = sim_scores.masked_fill(
                mask_inf,
                float('-inf')
            )
        sim_scores = sim_scores/math.sqrt(Eh)
        sim_scores = torch.softmax(sim_scores, dim=-1)

        cross_attention = sim_scores @ value_vecs # (B,H,T_dec,T_enc) @ (B,H,T_enc,Eh) = (B,H,T_dec,Eh)
        cross_attention = cross_attention.transpose(1,2) # (B,T_dec,H,Eh)
        cross_attention = cross_attention.reshape(B,T_dec,E) # (B,T_dec,E)
        output = self.o_w(cross_attention) # (B,T_dec,E) @ (E,E) = (B,T_dec,E)

        return output


## POSITIONAL ENCODINGS

In [53]:
def positionalEncoder(tensor):
    B,T,E = tensor.shape
    pos_encods = torch.zeros((1,T,E), device=tensor.device)
    for t in range(T) :
        for e in range(0,E,2) :
            pos_encods[:,t,e] = math.sin(t/math.pow(10000, e/E))
            pos_encods[:,t,e+1] = math.cos(t/math.pow(10000, e/E))
    return pos_encods


class AddPositionalEmbeds(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, tensor):
        B,T,E = tensor.shape
        pos_encods = positionalEncoder(tensor)
        return tensor + pos_encods
        
        

## Residual Connects , 1 DECODER BLOCK

In [54]:
class ResidualConnect(nn.Module):
    def __init__(self, sublayer):
        super().__init__()
        self.sublayer = sublayer
        
    def forward(self,x):
        return x + self.sublayer(x)

In [55]:
class DecoderBlockCrossAttention(nn.Module):
    def __init__(self, embed_dim, attention_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.MaskedAttention = MultiHeadAttention(embed_dim, attention_heads, masked=True)
        self.CrossAttention = CrossAttention(embed_dim, attention_heads)
        self.fnn = nn.Sequential(
            nn.Linear(embed_dim, 2048),
            nn.ReLU(),
            nn.Linear(2048,embed_dim)
        )
        self.layerNorm1 = nn.LayerNorm(self.embed_dim)
        self.layerNorm2 = nn.LayerNorm(self.embed_dim)
        self.layerNorm3 = nn.LayerNorm(self.embed_dim)

    def forward(self, input_embeds, batch_enc): 
        ## 1st Attention + Residual Connect + Norm
        o = input_embeds + self.MaskedAttention(input_embeds) # (B,T,E)
        o = self.layerNorm1(o) # (B,T,E)

        ## Cross Attention + Residual Connect + Norm
        o = o + self.CrossAttention(o, batch_enc)
        o = self.layerNorm2(o) # (B,T,E)

        o = o + self.fnn(o) # (B,T,E)
        o = self.layerNorm3(o) # (B,T,E)
        return o # (B,T,E)

# FULL DECODER 

## Decoder with Cross Attention

In [56]:
class DecoderWithCrossAttention(nn.Module):
    def __init__(self, vocab_count, embedding_dim,attention_heads, n_blocks):
        super().__init__()
        self.n_blocks = n_blocks
        self.add_positional = AddPositionalEmbeds()
        self.onn = nn.Sequential(
                    nn.Linear(embedding_dim, vocab_count), #(B,T,E)
                    # nn.Softmax(dim = -1)
        )
        self.encoder = SpatialEncoder(embedding_dim)
        self.embedding = nn.Embedding(vocab_count, embedding_dim)
        self.decoders = nn.ModuleList(DecoderBlockCrossAttention(embedding_dim, attention_heads) for _ in range(n_blocks))

        

    def forward(self,image_features, caption_input, caption_target):
        self.encoder.to(image_features.device)
        self.decoders.to(image_features.device)

        print(f'RECEIVED : image_features : {image_features.shape}, caption_input: {caption_input.shape}, caption_target : {caption_target.shape}')

        image_features = self.encoder(image_features) # B,T,C
        print(f'DULL ENCODER RETURNED  : {image_features.shape}')

        embeds = self.embedding(caption_input) # shape = (B,T,E)
        print(f'after 1st Embedding : {embeds.shape} ')

        embeds = self.add_positional(embeds) # shape = (B,T,E)
        print(f'after Positional Embedding : {embeds.shape} ')


        o = embeds # (B,T,E)
        
        for index,block in enumerate(self.decoders) :
            print('\n')
            print(f'HEAD : {index}')
            o = block(o,image_features) # (B,T,E)
            print(f'HEAD : {index} , output : {o.shape}')

        o = self.onn(o)
        return o
        
        

In [57]:
model = DecoderWithCrossAttention(len(caption_tokeniser.vocab),256,2,2)

In [58]:
a,b,c = next(iter(loader))
y = model(a,b,c)
y.shape

RECEIVED : image_features : torch.Size([32, 3, 224, 224]), caption_input: torch.Size([32, 21]), caption_target : torch.Size([32, 21])
ENCODER RECEIEVED SHAPE : torch.Size([32, 3, 224, 224])
SPATIAL-ENCODER RETURNED : torch.Size([32, 512, 7, 7])
ENCODER's Projection RETURNED : torch.Size([32, 256, 7, 7])
DULL ENCODER RETURNED  : torch.Size([32, 49, 256])
after 1st Embedding : torch.Size([32, 21, 256]) 
after Positional Embedding : torch.Size([32, 21, 256]) 


HEAD : 0
HEAD : 0 , output : torch.Size([32, 21, 256])


HEAD : 1
HEAD : 1 , output : torch.Size([32, 21, 256])


torch.Size([32, 21, 8256])